### Tools

Modesl can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of 

1 - A schema, including the name of the tool, a description and/or argument definitions (often a json schema)




2 - a function or coroutine to execute.

In [ ]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

response = model.invoke("name few models of openai")
response

AIMessage(content='Here are a handful of the most well‑known OpenAI models (both text and multimodal):\n\n| Category | Model | Typical use case(s) |\n|----------|-------|--------------------|\n| **Large language models (LLM)** | **GPT‑4** (including GPT‑4‑Vision) | Advanced conversational AI, content creation, summarization, reasoning, code generation, image‑to‑text tasks. |\n| | **GPT‑4 Turbo** | Same as GPT‑4 but faster and cheaper – the model behind ChatGPT’s current “Turbo” engine. |\n| | **GPT‑3.5 Turbo** | Earlier generation, still widely used for chat, coding, and general-purpose text generation. |\n| | **GPT‑3** (davinci, curie, babbage, ada) | Legacy models; davinci is the most capable, ada is fastest/cheapest. |\n| | **Codex** | Specialized for programming tasks (e.g., GitHub Copilot). |\n| **Multimodal / vision** | **DALL·E 3** | Text‑to‑image generation. |\n| | **Whisper** | Automatic speech‑to‑text transcription. |\n| | **CLIP** | Image‑text embedding for similarity search

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"Its sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("whats the weather in mumbai?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather function with location "Mumbai".', 'tool_calls': [{'id': 'fc_40148070-eeda-485c-9c43-c1bf92881b5f', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 128, 'total_tokens': 164, 'completion_time': 0.043479589, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006181846, 'prompt_tokens_details': None, 'queue_time': 0.367092273, 'total_time': 0.049661435}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_f102a3179c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1ecf3-0690-7250-81b1-2928457d3cc2-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'fc_40148070-eeda-485c-9c43-c1bf92881b5f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_toke

### Tool Execution loop


In [4]:
#Step 1 : Model generates tool calls

messages = [{"role": "user", "content": "what is the weather in delhi?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2 : exectue tools amd collect results

for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result= get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3 : Pass results back to model for final response 
final_response = model_with_tools.invoke(messages)

print(final_response.text)

It’s sunny in Delhi.
